# Demo: `ForecastingAssistant.compare()`

`compare()` backtests several forecaster/estimator configurations with the **same**
cross-validation strategy and returns a metric-ranked leaderboard, plus the winning
configuration as a reusable `BacktestResult`.

Key points:

- The dataset is profiled once and the profile is shared across every candidate.
- Each candidate goes through the existing `plan()` -> `backtest()` path, so every
  row carries a reproducible `code`.
- Ranking is a pure ascending metric sort (lower is better). The LLM is never involved.
- A failing candidate is recorded in the results table and sorted last; it never
  aborts the whole comparison.

This notebook covers a **single-series** example (`h2o`) and a **multi-series**
example (`items_sales`).

## Setup

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast
import skforecast_ai

print(skforecast.__version__)
print(skforecast_ai.__version__)

In [ ]:
import warnings

from skforecast.datasets import fetch_dataset
from skforecast.model_selection import TimeSeriesFold

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")

assistant = ForecastingAssistant()

## 1. Single-series example (`h2o`)

Monthly expenditure on corticosteroid drugs. The raw dataset has a target column
`y` and a timestamp column `datetime`.

In [ ]:
data = fetch_dataset(
    name="h2o", raw=True, kwargs_read={"names": ["y", "datetime"], "header": 0}
)
data.head()

### Explicit candidate configurations

Each candidate is a `(name, config)` tuple. The `config` dict accepts the same
override keys as `plan()`: `forecaster`, `estimator`, `estimator_kwargs`, `lags`,
and `window_features`.

All candidates are evaluated with the same `TimeSeriesFold`.

In [ ]:
cv = TimeSeriesFold(steps=12, initial_train_size=180, refit=False)

forecasters = [
    (
        "recursive_hgb",
        {
            "forecaster": "ForecasterRecursive",
            "estimator": "HistGradientBoostingRegressor",
        },
    ),
    (
        "recursive_ridge",
        {"forecaster": "ForecasterRecursive", "estimator": "Ridge"},
    ),
    (
        "direct_ridge",
        {
            "forecaster": "ForecasterDirect",
            "estimator": "Ridge",
            "lags": [1, 2, 3, 12],
        },
    ),
    (
        "foundation",
        {
            "forecaster": "ForecasterFoundation",
            "estimator_kwargs": {"device_map": "cpu"},
        },
    ),
]

comparison = assistant.compare(
    data=data,
    cv=cv,
    target="y",
    date_column="datetime",
    forecasters=forecasters,
    metric=["mean_absolute_scaled_error", "mean_absolute_error"],
)

comparison

### The ranked leaderboard

`results` is the ranked comparison table, one row per candidate, sorted best to
worst by `ranking_metric` (the first metric passed).

In [ ]:
print("Ranking metric:", comparison.ranking_metric)
comparison.results

### Inspect the winning configuration

`best_forecaster` is the top-ranked candidate as a full `BacktestResult`, carrying
its `profile`, `plan`, `metrics`, and reproducible `code`.

In [ ]:
best = comparison.best_forecaster

print("Best forecaster :", best.plan.forecaster)
print("Best estimator  :", best.plan.estimator)
print("Lags            :", best.plan.forecaster_kwargs.get("lags"))
best.metrics

In [ ]:
# Exact, runnable script for the winning configuration
print(best.code)

### Reuse the winner in other methods

Because the winning configuration is a full `BacktestResult`, its `profile` and
`plan` flow straight into `forecast()` (or `backtest()`, `forecast_code()`) with
no extra plumbing.

In [ ]:
forecast = assistant.forecast(
    data=data,
    steps=12,
    target="y",
    date_column="datetime",
    profile=best.profile,
    plan=best.plan,
)

forecast.predictions

### Zero-configuration comparison (`forecasters=None`)

When `forecasters` is omitted, the candidate set is built automatically from
`profile.forecaster_candidates`, each paired with its recommended estimator. This
gives a call that "just works".

Note: the auto set may include foundation and statistical forecasters, which
require their optional backends. Any candidate whose backend is missing is
recorded in the `error` column and ranked last instead of aborting the run.

In [ ]:
# comparison_auto = assistant.compare(
#     data=data,
#     cv=cv,
#     target="y",
#     date_column="datetime",
# )

# comparison_auto.results

## 2. Multi-series example (`items_sales`)

Simulated daily sales for 3 items in wide format (one column per series, indexed
by date). For multi-series tasks the leaderboard ranks candidates using the pooled
`average` metric across series.

In [ ]:
import gc
import torch

# (optional) drop big result objects you no longer need first:
# del comparison, comparison_auto

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
data = fetch_dataset(name="items_sales")
data.head()

In [ ]:
cv = TimeSeriesFold(steps=30, initial_train_size=900, refit=False)

forecasters = [
    (
        "multiseries_lgbm",
        {"forecaster": "ForecasterRecursiveMultiSeries", "estimator": "LGBMRegressor"},
    ),
    (
        "multiseries_hgb",
        {
            "forecaster": "ForecasterRecursiveMultiSeries",
            "estimator": "HistGradientBoostingRegressor",
        },
    ),
]

comparison_ms = assistant.compare(
    data=data,
    cv=cv,
    target=["item_1", "item_2", "item_3"],
    forecasters=forecasters,
)

comparison_ms

In [ ]:
print("Ranking metric:", comparison_ms.ranking_metric)
comparison_ms.results

In [ ]:
best_ms = comparison_ms.best_forecaster

print("Best forecaster:", best_ms.plan.forecaster)
print("Best estimator :", best_ms.plan.estimator)
best_ms.metrics